## Sel 1: Import Pustaka

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import warnings
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

## Sel 2: Memuat & Pra-pemrosesan Data (Resampling Harian)

In [2]:
# 1. BACA DATASET (Ganti 'dataset_jatim.csv' dengan nama file aslimu)
df = pd.read_csv('dataset_jatim.csv')

# --- BAGIAN INI HARUS DISESUAIKAN DENGAN NAMA KOLOM ASLI DI EXCEL-MU ---
kolom_waktu = 'Waktu'       # Kolom yang berisi tanggal dan jam
kolom_kota = 'Kota'         # Kolom nama kota/kabupaten
polutan = ['PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'CO (µg/m³)', 'NO2 (µg/m³)', 'Ozon (µg/m³)'] # Pastikan namanya persis sama
# -----------------------------------------------------------------------

# 2. Ubah format teks menjadi tipe Datetime yang dikenali Python
df[kolom_waktu] = pd.to_datetime(df[kolom_waktu])

# 3. Jadikan Waktu dan Kota sebagai Index agar mudah di-resample
df.set_index(kolom_waktu, inplace=True)

# 4. Resample: Hitung rata-rata polutan per HARI (huruf 'D' = Daily) untuk setiap Kota
# Cara ini akan memadatkan 24 baris (per jam) menjadi 1 baris per hari
df_daily = df.groupby(kolom_kota)[polutan].resample('D').mean().reset_index()

# Hapus baris yang kosong (misal ada hari dimana sensor mati total)
df_daily = df_daily.dropna()

print(f"Bentuk data setelah dijadikan rata-rata harian: {df_daily.shape}")
df_daily.head()

Bentuk data setelah dijadikan rata-rata harian: (80, 7)


,Kota,Waktu,PM2.5 (µg/m³),PM10 (µg/m³),CO (µg/m³),NO2 (µg/m³),Ozon (µg/m³)
0,Banyuwangi,2026-02-16,5.800000,7.825000,241.600000,0.435000,54.475000
1,Banyuwangi,2026-02-17,14.872083,16.853750,428.745417,1.504583,60.166250
2,Banyuwangi,2026-02-18,24.074583,26.213333,537.167917,3.636250,58.544583
3,Banyuwangi,2026-02-19,29.636250,32.924167,435.225833,1.957083,77.939167
4,Banyuwangi,2026-02-20,33.375000,36.002500,416.401667,0.831667,93.762083


## Sel 3: Rekayasa Fitur (Sliding Window 3 Hari & Target Besok)

In [3]:
# Fungsi untuk membuat Sliding Window (t-1, t-2, t-3) dan Target (t+1)
def create_sliding_window(df_kota, n_lags=3):
    df_temp = df_kota.copy()
    
    # Buat fitur history (t-1, t-2, t-3) untuk SETIAP polutan
    for p in polutan:
        for i in range(1, n_lags + 1):
            df_temp[f'{p}_H-{i}'] = df_temp[p].shift(i)
            
    # Buat target yang akan ditebak (t+1 alias BESOK)
    for p in polutan:
        df_temp[f'TARGET_{p}_Besok'] = df_temp[p].shift(-1)
        
    return df_temp

# Aplikasikan fungsi di atas untuk masing-masing kota secara terpisah agar tidak bentrok
df_model = df_daily.groupby(kolom_kota, group_keys=False).apply(create_sliding_window)

# Hapus baris yang memiliki NaN (akibat proses pergeseran hari/shift)
df_model = df_model.dropna().reset_index(drop=True)

# Lakukan One-Hot Encoding untuk kolom Kota (Mengubah teks kota menjadi kolom 0 dan 1)
df_model = pd.get_dummies(df_model, columns=[kolom_kota])

# ======================================================================
# BAGIAN YANG HILANG: MEMECAH DATA MENJADI X (Fitur) DAN y (Target)
# ======================================================================

# 1. Kumpulkan semua nama kolom yang merupakan jawaban (Target Besok)
kolom_target = [col for col in df_model.columns if 'TARGET_' in col]

# 2. Pisahkan tabel jawaban menjadi y
y = df_model[kolom_target]

# 3. Pisahkan tabel soal menjadi X 
# (Membuang kolom jawaban dan kolom 'Waktu' karena AI tidak bisa membaca tanggal)
X = df_model.drop(columns=kolom_target + [kolom_waktu])

print(f"Bentuk tabel awal df_model : {df_model.shape}")
print(f"Dimensi X (Fitur AI)       : {X.shape}")
print(f"Dimensi y (Target AI)      : {y.shape}")

Bentuk tabel awal df_model : (40, 36)
Dimensi X (Fitur AI)       : (40, 30)
Dimensi y (Target AI)      : (40, 5)


## Sel 4: Splitting & Training Model XGBoost

In [4]:
# 1. Splitting Data (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Memulai Hyperparameter Tuning XGBoost...")
print("Mohon tunggu, kipas laptop mungkin akan berputar lebih kencang...\n")

# 2. Inisiasi Model Dasar
model_xgb_dasar = MultiOutputRegressor(xgb.XGBRegressor(random_state=42))

# 3. Menentukan Ruang Pencarian Ruang (Grid)
param_grid = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'estimator__max_depth': [3, 5, 7, 9],
    'estimator__subsample': [0.8, 0.9, 1.0],
    'estimator__colsample_bytree': [0.8, 0.9, 1.0]
}

# 4. Mempersiapkan Mesin Pencari (Randomized Search)
grid_search = RandomizedSearchCV(
    estimator=model_xgb_dasar, 
    param_distributions=param_grid, 
    n_iter=15, # Akan mencoba 15 kombinasi acak terbaik
    cv=3,      # Menggunakan validasi silang 3 lipatan
    scoring='neg_mean_absolute_error', 
    random_state=42,
    n_jobs=-1, # Mengerahkan seluruh core prosesor agar cepat
    verbose=2  # Menampilkan log progres di layar
)

# 5. Eksekusi Pencarian dan Pelatihan
grid_search.fit(X_train, y_train)

# 6. Mengambil Otak Terbaik
model_terbaik = grid_search.best_estimator_

print("\n--- TUNING SELESAI ---")
print(f"Kombinasi Parameter Terbaik:\n{grid_search.best_params_}")

Memulai Hyperparameter Tuning XGBoost...
Mohon tunggu, kipas laptop mungkin akan berputar lebih kencang...

Fitting 3 folds for each of 15 candidates, totalling 45 fits

--- TUNING SELESAI ---
Kombinasi Parameter Terbaik:
{'estimator__subsample': 0.8, 'estimator__n_estimators': 300, 'estimator__max_depth': 3, 'estimator__learning_rate': 0.2, 'estimator__colsample_bytree': 1.0}


In [5]:
# Melakukan prediksi ke data ujian
y_pred = model_terbaik.predict(X_test)

# Daftar nama target (Sesuaikan jika urutan kolom y-mu berbeda)
polutan_labels = ['PM25', 'PM10', 'CO', 'NO2', 'O3'] 

print("--- EVALUASI MODEL TERBAIK PER POLUTAN ---")
for i, polutan in enumerate(polutan_labels):
    mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y_test.iloc[:, i], y_pred[:, i]))
    rata_rata_asli = y_test.iloc[:, i].mean()
    
    # Menghindari pembagian dengan nol jika rata-rata sangat kecil
    if rata_rata_asli > 0:
        persentase_error = (mae / rata_rata_asli) * 100
    else:
        persentase_error = 0.0
        
    print(f"\nPolutan {polutan}:")
    print(f"  - Rata-rata asli : {rata_rata_asli:.2f}")
    print(f"  - MAE (Meleset)  : {mae:.2f}")
    print(f"  - RMSE           : {rmse:.2f}")
    print(f"  - Error (%)      : {persentase_error:.2f}%")

--- EVALUASI MODEL TERBAIK PER POLUTAN ---

Polutan PM25:
  - Rata-rata asli : 15.77
  - MAE (Meleset)  : 5.08
  - RMSE           : 7.82
  - Error (%)      : 32.23%

Polutan PM10:
  - Rata-rata asli : 17.74
  - MAE (Meleset)  : 4.99
  - RMSE           : 7.59
  - Error (%)      : 28.13%

Polutan CO:
  - Rata-rata asli : 308.14
  - MAE (Meleset)  : 68.89
  - RMSE           : 107.27
  - Error (%)      : 22.36%

Polutan NO2:
  - Rata-rata asli : 1.62
  - MAE (Meleset)  : 0.91
  - RMSE           : 2.06
  - Error (%)      : 55.76%

Polutan O3:
  - Rata-rata asli : 54.28
  - MAE (Meleset)  : 8.71
  - RMSE           : 16.78
  - Error (%)      : 16.04%


## Sel 5: Ekspor Model (Menyimpan Hasil)

In [6]:
# Membungkus model dan nama kolom agar Backend tidak kebingungan
paket_model = {
    'model': model_terbaik,
    'fitur': list(X_train.columns)
}

# Menyimpan ke dalam file Pickle
nama_file = 'xgb_ispu_jatim.pkl'
joblib.dump(paket_model, nama_file)

print(f"Selesai! Model pintar telah berhasil diekspor sebagai '{nama_file}'")
print("Jangan lupa salin file ini dan timpa file lama yang ada di dalam folder 'backend/models/'!")

Selesai! Model pintar telah berhasil diekspor sebagai 'xgb_ispu_jatim.pkl'
Jangan lupa salin file ini dan timpa file lama yang ada di dalam folder 'backend/models/'!
